In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

def preprocess_for_anomaly(ds, varname="O3", lat_range=(60, 90), months=[3, 4, 5]):
    """
    Preprocess the dataset to obtain a 2D (pressure x time) climatological field for O3.
    
    Steps:
      1) Average over longitude (if available).
      2) Average over the 'member' dimension (if available).
      3) Select the specified latitude range and perform a cosine(latitude)-weighted average.
      4) Select data for the specified months (e.g., Mar, Apr, May) and average over all years (groupby time.month).
      5) Rename the resulting 'month' dimension to 'time' and assign dummy dates (e.g., the 15th of each month in 2000)
         so that the x-axis in plots can display month names.
    
    Returns:
      time_vals: Dummy time coordinate (a datetime array corresponding to each month).
      plev: Pressure coordinate.
      da_clim: 2D DataArray with dimensions (plev, time).
    """
    if ds is None or varname not in ds:
        return None, None, None

    da = ds[varname]
    # 1) Average over lon if present
    if "lon" in da.dims:
        da = da.mean(dim="lon")
    # 2) Average over member if present
    if "member" in da.dims:
        da = da.mean(dim="member")
    # 3) Select specified latitudes and perform cosine(latitude) weighting
    if "lat" not in da.dims:
        return None, None, None
    da = da.sel(lat=slice(lat_range[0], lat_range[1]))
    weights = np.cos(np.deg2rad(da.lat))
    da = da.weighted(weights).mean(dim="lat")
    # Check for pressure dimension
    if "plev" not in da.dims:
        return None, None, None
    # 4) Select specified months
    da = da.sel(time=da.time.dt.month.isin(months))
    # Group by month and average over all years
    da_clim = da.groupby("time.month").mean(dim="time")
    da_clim = da_clim.sortby("month")
    # 5) Create dummy time coordinates (e.g., 15th of each month in 2000)
    month_vals = da_clim["month"].values
    time_dummy = np.array([np.datetime64(f"2000-{int(m):02d}-15") for m in month_vals])
    da_clim = da_clim.rename({"month": "time"})
    da_clim = da_clim.assign_coords(time=time_dummy)
    # Transpose to (plev, time)
    da_clim = da_clim.transpose("plev", "time")
    return da_clim.time, da_clim.plev, da_clim

def open_restart_run_data(base_dir, run, year="0008"):
    """
    Open netCDF files for the specified run (e.g., 'Jan', 'Feb', 'Mar', 'Apr', 'Feb_2', etc.)
    and return the merged xarray.Dataset. Only time data for months 1-5 is selected.
    """
    # Mapping for base run names to month strings
    run2month = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05"}
    # For runs like "Feb_2", "Mar_3", etc., extract the base run name
    run_key = run.split('_')[0]
    month_str = run2month.get(run_key, "01")
    
    file_pattern = os.path.join(
        base_dir, run,
        f"BWCN.e122.f19_g16.002.{year}-{month_str}.*.nc"
    )
    
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    ds = ds.sel(time=ds.time.dt.month <= 5)
    return ds

def get_common_times(time_arrays):
    """
    Given a list of numpy arrays (each containing np.datetime64 values),
    return the sorted array of common time points (intersection of all arrays).
    """
    common = set(time_arrays[0].tolist())
    for t in time_arrays[1:]:
        common = common.intersection(set(t.tolist()))
    common = sorted(list(common))
    return np.array(common)

def trim_to_common_times(time_array, data_array, common_times):
    """
    Trim the time_array and the corresponding 2D data_array (with time as the second dimension)
    so that only the time points present in common_times are retained.
    """
    times = time_array.values
    mask = np.isin(times, common_times)
    trimmed_time = time_array[mask]
    trimmed_data = data_array[:, mask]
    return trimmed_time, trimmed_data

def get_subplot_positions(n):
    """
    Return a list of axes positions (in normalized figure coordinates) for n subplots,
    arranged in a fixed 2-row by 3-column grid with custom positions.
    
    For n == 4, the positions (row, col) are:
      (1, 1.5), (2, 1.5), (1, 2.5), (2, 2.5)
    For n == 5, the positions (row, col) are:
      (1, 1), (1, 2), (1, 3), (2, 1.5), (2, 2.5)
      
    This version further increases margins to add more spacing.
    """
    # Further increased margins
    left_margin = 0.07
    right_margin = 0.88
    bottom_margin = 0.12
    top_margin = 0.88
    width_avail = right_margin - left_margin   # 0.81
    height_avail = top_margin - bottom_margin  # 0.76

    # For a 3-column grid, each cell width:
    cell_width = width_avail / 3               # ~0.27
    # For a 2-row grid, each cell height:
    cell_height = height_avail / 2             # ~0.38

    # Compute centers for integer columns:
    col1_center = left_margin + 0.5 * cell_width    # ~0.07 + 0.135 = 0.205
    col2_center = left_margin + 1.5 * cell_width    # ~0.07 + 0.405 = 0.475
    col3_center = left_margin + 2.5 * cell_width    # ~0.07 + 0.675 = 0.745
    # Compute centers for fractional columns:
    col1_5_center = (col1_center + col2_center) / 2  # ~0.34
    col2_5_center = (col2_center + col3_center) / 2  # ~0.61

    # Compute row centers:
    row1_center = bottom_margin + cell_height + 0.5 * cell_height  # ~0.12 + 0.38 + 0.19 = 0.69
    row2_center = bottom_margin + 0.5 * cell_height                # ~0.12 + 0.19 = 0.31

    # Slightly smaller subplots for more spacing
    subplot_width = 0.20
    subplot_height = 0.32

    positions = []
    if n == 4:
        # Grid coordinates for 4 subplots: (row, col)
        grid_coords = [(1, 1.5), (2, 1.5), (1, 2.5), (2, 2.5)]
    elif n == 5:
        # Grid coordinates for 5 subplots: (row, col)
        grid_coords = [(1, 1), (1, 2), (1, 3), (2, 1.5), (2, 2.5)]
    else:
        # Default: fill up to n subplots in a 2x3 grid
        grid_coords = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]
        grid_coords = grid_coords[:n]
    
    for row, col in grid_coords:
        if row == 1:
            center_y = row1_center
        elif row == 2:
            center_y = row2_center
        else:
            center_y = row1_center  # fallback

        # Determine center_x based on col value
        if col == 1:
            center_x = col1_center
        elif col == 2:
            center_x = col2_center
        elif col == 3:
            center_x = col3_center
        elif col == 1.5:
            center_x = col1_5_center
        elif col == 2.5:
            center_x = col2_5_center
        else:
            center_x = col2_center  # fallback

        left_pos = center_x - subplot_width / 2
        bottom_pos = center_y - subplot_height / 2
        positions.append([left_pos, bottom_pos, subplot_width, subplot_height])
    return positions

def plot_composite(runs_data, composite_title, subtitle, save_path):
    """
    Plot a composite figure for multiple runs using a fixed 2x3 grid layout with custom positions.
    
    The function:
      - Uses common time points (from the intersection of all runs) to set the x-axis ticks.
      - Removes the x-axis title.
      - Sets the x-axis limits to the common time range so that the ticks start from the left edge.
    """
    # Determine common time axis from all runs
    time_arrays = [rd[1] for rd in runs_data if rd[1] is not None]
    if not time_arrays:
        raise ValueError("No valid time arrays found.")
    common_times = get_common_times([t.values for t in time_arrays])
    if len(common_times) == 0:
        raise ValueError("No common time points found among runs.")
    
    # Compute global anomaly range for a unified color scale (symmetric about zero)
    all_anom = []
    for _, t, _, anom in runs_data:
        if anom is not None:
            mask = np.isin(t.values, common_times)
            if np.any(mask):
                all_anom.append(anom.values[:, mask].ravel())
    if not all_anom:
        raise ValueError("No anomaly data found.")
    combined_anom = np.concatenate(all_anom)
    max_abs = np.nanmax(np.abs(combined_anom))
    vmin, vmax = -max_abs, max_abs

    n_subplots = len(runs_data)
    positions = get_subplot_positions(n_subplots)
    
    # Create a larger figure to allow more space
    fig = plt.figure(figsize=(14, 9))
    
    # Pre-calculate common x-axis tick positions and labels
    common_ticks = mdates.date2num(common_times)
    tick_labels = [mdates.num2date(tick).strftime('%b') for tick in common_ticks]
    
    cf_last = None
    for i, (label, time_arr, plev, anom) in enumerate(runs_data):
        ax = fig.add_axes(positions[i])
        if time_arr is not None and anom is not None:
            trimmed_time, trimmed_anom = trim_to_common_times(time_arr, anom, common_times)
            time_mpl = mdates.date2num(trimmed_time.values)
            cf_last = ax.contourf(time_mpl, plev, trimmed_anom, levels=20, cmap="RdBu_r",
                                  vmin=vmin, vmax=vmax)
        ax.set_yscale("log")
        ax.invert_yaxis()
        # Set x-axis ticks based on common_times (one tick per month)
        ax.set_xticks(common_ticks)
        ax.set_xticklabels(tick_labels, fontsize=9)
        # Set x-axis limits to span the common time range (start from the left edge)
        ax.set_xlim(common_ticks[0], common_ticks[-1])
        # Remove x-axis title by not setting xlabel
        ax.set_ylabel("Pressure (Pa)", fontsize=9)
        ax.set_title(label, fontsize=10)
        ax.set_ylim(1e5, 1e1)
    
    # Add a unified colorbar on the right side of the figure
    cax = fig.add_axes([0.91, 0.25, 0.02, 0.5])  # colorbar position also shifted
    if cf_last is not None:
        fig.colorbar(cf_last, cax=cax, label="O3 Anomaly [mol/mol]")
    
    fig.suptitle(composite_title + "\n" + subtitle, fontsize=14)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"[INFO] Saved composite figure: {save_path}")

if __name__ == "__main__":
    # Define base directory and reference file path
    base_dir = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008"
    ref_file = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc"
    
    # Compute background climatology using the merged file (selecting Mar, Apr, May)
    ds_merged = xr.open_dataset(ref_file)
    bg_time, bg_plev, bg_o3 = preprocess_for_anomaly(ds_merged, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    
    # Process REF run: select year "0008" for Mar, Apr, May
    ds_ref = xr.open_dataset(ref_file)
    ds_ref_0008 = ds_ref.sel(time=(ds_ref.time.dt.year == 8) & (ds_ref.time.dt.month.isin([3, 4, 5])))
    ref_time, ref_plev, ref_clim = preprocess_for_anomaly(ds_ref_0008, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_ref = ref_clim - bg_o3
    
    # Process Jan run
    ds_jan = open_restart_run_data(base_dir, run="Jan", year="0008")
    jan_time, jan_plev, jan_clim = preprocess_for_anomaly(ds_jan, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_jan = jan_clim - bg_o3
    
    # Process Feb run
    ds_feb = open_restart_run_data(base_dir, run="Feb", year="0008")
    feb_time, feb_plev, feb_clim = preprocess_for_anomaly(ds_feb, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_feb = feb_clim - bg_o3
    
    # Process Mar run
    ds_mar = open_restart_run_data(base_dir, run="Mar", year="0008")
    mar_time, mar_plev, mar_clim = preprocess_for_anomaly(ds_mar, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_mar = mar_clim - bg_o3
    
    # Process Apr run
    ds_apr = open_restart_run_data(base_dir, run="Apr", year="0008")
    apr_time, apr_plev, apr_clim = preprocess_for_anomaly(ds_apr, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_apr = apr_clim - bg_o3
    
    # Process Feb_2 run
    ds_feb2 = open_restart_run_data(base_dir, run="Feb_2", year="0008")
    feb2_time, feb2_plev, feb2_clim = preprocess_for_anomaly(ds_feb2, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_feb2 = feb2_clim - bg_o3
    
    # Process Feb_3 run
    ds_feb3 = open_restart_run_data(base_dir, run="Feb_3", year="0008")
    feb3_time, feb3_plev, feb3_clim = preprocess_for_anomaly(ds_feb3, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_feb3 = feb3_clim - bg_o3
    
    # Process Mar_2 run
    ds_mar2 = open_restart_run_data(base_dir, run="Mar_2", year="0008")
    mar2_time, mar2_plev, mar2_clim = preprocess_for_anomaly(ds_mar2, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_mar2 = mar2_clim - bg_o3
    
    # Process Mar_3 run
    ds_mar3 = open_restart_run_data(base_dir, run="Mar_3", year="0008")
    mar3_time, mar3_plev, mar3_clim = preprocess_for_anomaly(ds_mar3, varname="O3", lat_range=(60, 90), months=[3, 4, 5])
    anom_mar3 = mar3_clim - bg_o3
    
    # ---------------- Composite Plot 1 ----------------
    # Composite: REF, Jan, Feb, Mar, Apr with subtitle "small pertlim"
    runs_composite1 = [
        ("REF", ref_time, ref_plev, anom_ref),
        ("Jan", jan_time, jan_plev, anom_jan),
        ("Feb", feb_time, feb_plev, anom_feb),
        ("Mar", mar_time, mar_plev, anom_mar),
        ("Apr", apr_time, apr_plev, anom_apr)
    ]
    composite_title1 = "Composite O3 Anomaly (REF, Jan, Feb, Mar, Apr)"
    subtitle1 = "small pertlim"
    save_path1 = "/home/weiji/restart_exam/plots/O3VERTICAL/composite_REF_Jan_Feb_Mar_Apr.png"
    plot_composite(runs_composite1, composite_title1, subtitle1, save_path1)
    
    # ---------------- Composite Plot 2 ----------------
    # Composite: REF, Feb, Feb_2, Feb_3 with subtitle "Feb initial different pertlim"
    # Subplot labels: "REF", "F (small pertlim)", "F2 (large pertlim)", "F3 (humidity pertlim)"
    runs_composite2 = [
        ("REF", ref_time, ref_plev, anom_ref),
        ("F (small pertlim)", feb_time, feb_plev, anom_feb),
        ("F2 (large pertlim)", feb2_time, feb2_plev, anom_feb2),
        ("F3 (humidity pertlim)", feb3_time, feb3_plev, anom_feb3)
    ]
    composite_title2 = "Composite O3 Anomaly (REF, Feb variants)"
    subtitle2 = "Feb initial different pertlim"
    save_path2 = "/home/weiji/restart_exam/plots/O3VERTICAL/composite_REF_Feb_variants.png"
    plot_composite(runs_composite2, composite_title2, subtitle2, save_path2)
    
    # ---------------- Composite Plot 3 ----------------
    # Composite: REF, m2 (large pertlim), m3 (humidity pertlim), march initial (small pertlim)
    runs_composite3 = [
        ("REF", ref_time, ref_plev, anom_ref),
        ("m2 (large pertlim)", mar2_time, mar2_plev, anom_mar2),
        ("m3 (humidity pertlim)", mar3_time, mar3_plev, anom_mar3),
        ("march initial (small pertlim)", mar_time, mar_plev, anom_mar)
    ]
    composite_title3 = "Composite O3 Anomaly (REF, Mar variants)"
    subtitle3 = "Mar initial different pertlim"
    save_path3 = "/home/weiji/restart_exam/plots/O3VERTICAL/composite_REF_Mar_variants.png"
    plot_composite(runs_composite3, composite_title3, subtitle3, save_path3)
    
    print("[INFO] All composite figures have been generated.")
